In [1]:
from pathlib import Path
from dotenv import load_dotenv

for _dir in [Path.cwd(), *Path.cwd().parents]:
    _env = _dir / '.env'
    if _env.is_file():
        load_dotenv(_env)
        break

In [2]:
from databricks.connect import DatabricksSession

spark = DatabricksSession.builder.getOrCreate()
# spark.sql('CREATE SCHEMA IF NOT EXISTS ddc_databricks.silver')

DataFrame[]

In [3]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StringType, IntegerType, LongType, DoubleType, FloatType,
    BooleanType, TimestampType, DateType, ShortType, ByteType, DecimalType,
)
from datetime import datetime

CATALOG = 'ddc_databricks'
SCHEMA  = 'silver'

SILVER_TABLES = [
    'github_repos',
    'github_contributors',
    'github_contributor_stats',
    'github_events',
    'github_event_features',
    'github_readmes',
    'github_package_features',
    'pypi_metadata',
    'pypi_downloads_recent',
    'pypi_downloads_overall_daily',
    'pypi_downloads_overall_features',
    'pypi_package_features',
    'so_questions',
    'so_answers',
    'so_tag_info',
    'so_package_features',
]

NUMERIC_TYPES = (
    IntegerType, LongType, DoubleType, FloatType, ShortType, ByteType, DecimalType,
)
STRING_TYPES = (StringType,)

run_ts = datetime.utcnow().isoformat()
print(f'Profiling {len(SILVER_TABLES)} silver tables  (run: {run_ts})')

Profiling 16 silver tables  (run: 2026-03-27T18:45:51.481998)


C:\Users\Behroz Karim\AppData\Local\Temp\ipykernel_99872\3387527843.py:35: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  run_ts = datetime.utcnow().isoformat()


In [4]:
table_stats_rows = []

for tbl_name in SILVER_TABLES:
    fqn = f'{CATALOG}.{SCHEMA}.{tbl_name}'
    try:
        df = spark.table(fqn)
    except Exception as exc:
        print(f'  SKIP {fqn}: {exc}')
        continue

    row_count = df.count()
    col_count = len(df.columns)

    numeric_cols = [f.name for f in df.schema.fields if isinstance(f.dataType, NUMERIC_TYPES)]
    string_cols  = [f.name for f in df.schema.fields if isinstance(f.dataType, STRING_TYPES)]
    bool_cols    = [f.name for f in df.schema.fields if isinstance(f.dataType, BooleanType)]
    ts_cols      = [f.name for f in df.schema.fields if isinstance(f.dataType, (TimestampType, DateType))]

    total_nulls = 0
    if row_count > 0:
        null_exprs = [F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c) for c in df.columns]
        null_row = df.select(null_exprs).collect()[0]
        total_nulls = sum(null_row[c] or 0 for c in df.columns)

    completeness = round(1.0 - total_nulls / (row_count * col_count), 4) if row_count * col_count > 0 else 0.0

    duplicate_rows = row_count - df.dropDuplicates().count() if row_count > 0 else 0

    table_stats_rows.append((
        tbl_name, row_count, col_count,
        len(numeric_cols), len(string_cols), len(bool_cols), len(ts_cols),
        total_nulls, completeness, duplicate_rows, run_ts,
    ))
    print(f'  {tbl_name}: {row_count:,} rows, {col_count} cols, completeness={completeness}')

table_stats_df = spark.createDataFrame(
    table_stats_rows,
    ['table_name', 'row_count', 'column_count',
     'numeric_columns', 'string_columns', 'boolean_columns', 'timestamp_columns',
     'total_null_values', 'completeness_ratio', 'duplicate_row_count', 'profiled_at'],
)

table_stats_df.show(truncate=False)

  github_repos: 8 rows, 24 cols, completeness=1.0
  github_contributors: 800 rows, 8 cols, completeness=1.0
  github_contributor_stats: 8 rows, 6 cols, completeness=1.0
  github_events: 100 rows, 15 cols, completeness=0.676
  github_event_features: 0 rows, 10 cols, completeness=0.0
  github_readmes: 8 rows, 17 cols, completeness=1.0
  github_package_features: 8 rows, 46 cols, completeness=0.7826
  pypi_metadata: 1 rows, 29 cols, completeness=0.4828
  pypi_downloads_recent: 8 rows, 9 cols, completeness=1.0
  pypi_downloads_overall_daily: 2,896 rows, 7 cols, completeness=1.0
  pypi_downloads_overall_features: 8 rows, 10 cols, completeness=1.0
  pypi_package_features: 8 rows, 36 cols, completeness=0.5
  so_questions: 2,400 rows, 29 cols, completeness=0.9261
  so_answers: 4,800 rows, 22 cols, completeness=0.9866
  so_tag_info: 8 rows, 10 cols, completeness=1.0
  so_package_features: 8 rows, 23 cols, completeness=1.0
+-------------------------------+---------+------------+---------------+--

In [5]:
column_stats_rows = []

for tbl_name in SILVER_TABLES:
    fqn = f'{CATALOG}.{SCHEMA}.{tbl_name}'
    try:
        df = spark.table(fqn)
    except Exception:
        continue

    row_count = df.count()
    if row_count == 0:
        continue

    for field in df.schema.fields:
        col_name  = field.name
        col_dtype = field.dataType.simpleString()
        is_nullable = field.nullable

        base_aggs = [
            F.count(F.when(F.col(col_name).isNull(), True)).alias('null_count'),
            F.countDistinct(F.col(col_name)).alias('distinct_count'),
        ]

        num_min = num_max = num_mean = num_stddev = None
        str_min_len = str_max_len = str_avg_len = str_empty_count = None

        if isinstance(field.dataType, NUMERIC_TYPES):
            base_aggs += [
                F.min(F.col(col_name).cast('double')).alias('num_min'),
                F.max(F.col(col_name).cast('double')).alias('num_max'),
                F.round(F.avg(F.col(col_name).cast('double')), 4).alias('num_mean'),
                F.round(F.stddev(F.col(col_name).cast('double')), 4).alias('num_stddev'),
            ]
        elif isinstance(field.dataType, STRING_TYPES):
            base_aggs += [
                F.min(F.length(F.col(col_name))).alias('str_min_len'),
                F.max(F.length(F.col(col_name))).alias('str_max_len'),
                F.round(F.avg(F.length(F.col(col_name)).cast('double')), 2).alias('str_avg_len'),
                F.sum(F.when(
                    (F.col(col_name).isNull()) | (F.trim(F.col(col_name)) == ''), 1
                ).otherwise(0)).alias('str_empty_count'),
            ]

        stats_row = df.select(base_aggs).collect()[0]

        null_count     = int(stats_row['null_count'])
        distinct_count = int(stats_row['distinct_count'])
        null_pct       = round(null_count / row_count * 100, 2)
        distinct_pct   = round(distinct_count / row_count * 100, 2)
        uniqueness     = 1.0 if distinct_count == row_count else round(distinct_count / row_count, 4)

        if isinstance(field.dataType, NUMERIC_TYPES):
            num_min    = stats_row['num_min']
            num_max    = stats_row['num_max']
            num_mean   = stats_row['num_mean']
            num_stddev = stats_row['num_stddev']
        elif isinstance(field.dataType, STRING_TYPES):
            str_min_len     = stats_row['str_min_len']
            str_max_len     = stats_row['str_max_len']
            str_avg_len     = float(stats_row['str_avg_len']) if stats_row['str_avg_len'] is not None else None
            str_empty_count = int(stats_row['str_empty_count'])

        column_stats_rows.append((
            tbl_name, col_name, col_dtype, is_nullable,
            null_count, null_pct, distinct_count, distinct_pct, uniqueness,
            num_min, num_max, num_mean, num_stddev,
            str_min_len, str_max_len, str_avg_len, str_empty_count,
            run_ts,
        ))

    print(f'  {tbl_name}: {len(df.schema.fields)} columns profiled')

print(f'\nTotal column profiles: {len(column_stats_rows)}')

  github_repos: 24 columns profiled
  github_contributors: 8 columns profiled
  github_contributor_stats: 6 columns profiled
  github_events: 15 columns profiled
  github_readmes: 17 columns profiled
  github_package_features: 46 columns profiled
  pypi_metadata: 29 columns profiled
  pypi_downloads_recent: 9 columns profiled
  pypi_downloads_overall_daily: 7 columns profiled
  pypi_downloads_overall_features: 10 columns profiled
  pypi_package_features: 36 columns profiled
  so_questions: 29 columns profiled
  so_answers: 22 columns profiled
  so_tag_info: 10 columns profiled
  so_package_features: 23 columns profiled

Total column profiles: 291


In [6]:
column_stats_df = spark.createDataFrame(
    column_stats_rows,
    ['table_name', 'column_name', 'data_type', 'is_nullable',
     'null_count', 'null_pct', 'distinct_count', 'distinct_pct', 'uniqueness',
     'num_min', 'num_max', 'num_mean', 'num_stddev',
     'str_min_len', 'str_max_len', 'str_avg_len', 'str_empty_count',
     'profiled_at'],
)

column_stats_df.filter(F.col('table_name') == 'github_repos').show(50, truncate=False)

+------------+----------------+-------------+-----------+----------+--------+--------------+------------+----------+--------+------------+-------------+--------------+-----------+-----------+-----------+---------------+--------------------------+
|table_name  |column_name     |data_type    |is_nullable|null_count|null_pct|distinct_count|distinct_pct|uniqueness|num_min |num_max     |num_mean     |num_stddev    |str_min_len|str_max_len|str_avg_len|str_empty_count|profiled_at               |
+------------+----------------+-------------+-----------+----------+--------+--------------+------------+----------+--------+------------+-------------+--------------+-----------+-----------+-----------+---------------+--------------------------+
|github_repos|package_name    |string       |true       |0         |0.0     |8             |100.0       |1.0       |NULL    |NULL        |NULL         |NULL          |5          |12         |6.75       |0              |2026-03-27T18:45:51.481998|
|github_repo

In [7]:
(
    table_stats_df
    .write
    .format('delta')
    .mode('append')
    .saveAsTable(f'{CATALOG}.{SCHEMA}.profiling_table_stats')
)
print('silver.profiling_table_stats written (append mode)')

(
    column_stats_df
    .write
    .format('delta')
    .mode('append')
    .saveAsTable(f'{CATALOG}.{SCHEMA}.profiling_column_stats')
)
print('silver.profiling_column_stats written (append mode)')

silver.profiling_table_stats written (append mode)
silver.profiling_column_stats written (append mode)


In [8]:
print('SILVER PROFILING SUMMARY')

total_rows = table_stats_df.select(F.sum('row_count')).collect()[0][0]
total_cols = column_stats_df.count()

print(f'Run timestamp       : {run_ts}')
print(f'Tables profiled     : {len(table_stats_rows)}')
print(f'Total rows (all)    : {total_rows:,}')
print(f'Column profiles     : {total_cols}')
print()

print('Tables with lowest completeness')
table_stats_df.orderBy('completeness_ratio').select(
    'table_name', 'row_count', 'completeness_ratio', 'duplicate_row_count',
).show(5, truncate=False)

print('Columns with highest null percentage')
column_stats_df.orderBy(F.col('null_pct').desc()).select(
    'table_name', 'column_name', 'data_type', 'null_pct', 'distinct_count',
).show(15, truncate=False)

print('Columns with low cardinality (potential flags/enums)')
column_stats_df.filter(
    (F.col('distinct_count') <= 10) & (F.col('distinct_count') > 0)
).orderBy('distinct_count').select(
    'table_name', 'column_name', 'data_type', 'distinct_count', 'null_pct',
).show(15, truncate=False)

print('Numeric columns with highest stddev (most variable)')
column_stats_df.filter(
    F.col('num_stddev').isNotNull()
).orderBy(F.col('num_stddev').desc()).select(
    'table_name', 'column_name', 'num_min', 'num_max', 'num_mean', 'num_stddev',
).show(10, truncate=False)

print('\nDone. Tables written:')
print('  silver.profiling_table_stats   (append)')
print('  silver.profiling_column_stats  (append)')

SILVER PROFILING SUMMARY
Run timestamp       : 2026-03-27T18:45:51.481998
Tables profiled     : 16
Total rows (all)    : 11,069
Column profiles     : 291

Tables with lowest completeness
+-----------------------+---------+------------------+-------------------+
|table_name             |row_count|completeness_ratio|duplicate_row_count|
+-----------------------+---------+------------------+-------------------+
|github_event_features  |0        |0.0               |0                  |
|pypi_metadata          |1        |0.4828            |0                  |
|pypi_package_features  |8        |0.5               |0                  |
|github_events          |100      |0.676             |0                  |
|github_package_features|8        |0.7826            |0                  |
+-----------------------+---------+------------------+-------------------+
only showing top 5 rows
Columns with highest null percentage
+-----------------------+---------------------+---------+--------+-----------